# Motor de Voz Ultra Realista (ChatTTS) para Miniverse
Este cuaderno permite correr **ChatTTS** en Google Colab y exponerlo mediante un túnel (ngrok) para conectarlo a tu backend local.

### ⚠️ PASO MUY IMPORTANTE (NO OMITIR):
Asegúrate de ir al menú superior: **Entorno de Ejecución > Cambiar tipo de entorno de ejecución** y elige **T4 GPU**. (Si no lo haces, correrá en CPU y será lentísimo).


In [ ]:
!pip install fastapi uvicorn pyngrok soundfile
!pip install ChatTTS

### Crear el servidor API
Esta celda guarda el código en un archivo interno para evitar errores de bucles de ejecución.

In [ ]:
%%writefile app.py
from fastapi import FastAPI
from fastapi.responses import FileResponse
from pydantic import BaseModel
import soundfile as sf
import tempfile
import warnings
warnings.filterwarnings('ignore')

import ChatTTS
import torch

app = FastAPI()
print("Cargando modelo ChatTTS hiperrealista...")
chat = ChatTTS.Chat()
chat.load(compile=False)
print("Modelo cargado y listo.")

# Generamos dos voces distintas para diferenciar sexos
torch.manual_seed(1111)
voice_female = chat.sample_random_speaker()
torch.manual_seed(2222)
voice_male = chat.sample_random_speaker()

class VoiceRequest(BaseModel):
    text: str
    sexo: str = "neutral"
    language: str = "es"

@app.post("/synthesize")
def synthesize(req: VoiceRequest):
    print(f"Generando audio para: {req.text[:50]}...")
    spk = voice_female if req.sexo.lower() == "femenino" else voice_male
    params_infer_code = ChatTTS.Chat.InferCodeParams(spk_emb=spk)
    
    wavs = chat.infer([req.text], params_infer_code=params_infer_code)
    
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(tmp.name, wavs[0][0], 24000)
    return FileResponse(tmp.name, media_type="audio/wav")


### Iniciar Servidor y Ngrok
Pon tu token de Ngrok, ejecuta esta celda y pon la URL en tu `.env` local.

In [ ]:
import os
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "TU_NGROK_AUTH_TOKEN_AQUI" # <--- PON TU TOKEN AQUÍ
os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN
!ngrok config add-authtoken $NGROK_AUTH_TOKEN

public_url = ngrok.connect(8000).public_url
print("\n" + "="*60)
print(f"API PUBLICA ACTIVA EN: {public_url}")
print("Pon esta URL en tu .env local: CHATTERBOX_REMOTE_URL=", public_url)
print("="*60 + "\n")

!uvicorn app:app --host 0.0.0.0 --port 8000